# 🛞 Clasificación de Llantas Dañadas — RNAPD 2026-I
**Pregunta 1 | Ph.D. Aldo Camargo**

Este notebook ejecuta **todo el pipeline** en Google Colab (GPU T4):
1. Instalación de dependencias
2. Descarga y preparación del dataset (Kaggle)
3. Análisis del desbalance de clases
4. Entrenamiento: CNN desde cero · ResNet-50 · EfficientNet-B0
5. Estudio de ablación
6. Evaluación comparativa (métricas, ROC, matrices de confusión)
7. Interpretabilidad Grad-CAM
8. Análisis cualitativo de fallos

> **Antes de ejecutar**: Runtime → Change runtime type → **GPU (T4)**

## 📦 Celda 1 — Instalación y configuración

In [ ]:
# ── Instalación ──────────────────────────────────────────────────────────────
!pip install -q kaggle scikit-learn seaborn matplotlib torchvision

import os, sys, json, random, shutil, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import datasets, transforms, models
from pathlib import Path
from PIL import Image
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
warnings.filterwarnings('ignore')

# ── Reproducibilidad ─────────────────────────────────────────────────────────
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# ── Constantes globales ───────────────────────────────────────────────────────
DATA_DIR   = Path('data/tire_texture')
RESULTS    = Path('results')
IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS     = 30
PATIENCE   = 7
RESULTS.mkdir(parents=True, exist_ok=True)
print('✅ Dependencias cargadas.')

## 📥 Celda 2 — Descarga del Dataset (Kaggle)

In [ ]:
# ── Configurar credenciales Kaggle ────────────────────────────────────────────
# Opción A: subir kaggle.json manualmente
from google.colab import files
print('Sube tu kaggle.json (descárgalo desde kaggle.com → Account → API → Create New Token)')
uploaded = files.upload()

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.move('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

# ── Descargar y descomprimir ──────────────────────────────────────────────────
!kaggle datasets download -d jehanbhathena/tire-texture-image-recognition -p /content/kaggle_raw
!unzip -q /content/kaggle_raw/tire-texture-image-recognition.zip -d /content/kaggle_raw/
print('✅ Dataset descargado.')

## 🗂️ Celda 3 — Preparación del Dataset

In [ ]:
# ── Mapeo categorías Kaggle → binario ─────────────────────────────────────────
CATEGORY_MAP = {
    'good': 'good_tire', 'normal': 'good_tire',
    'cracked': 'damaged_tire', 'broken': 'damaged_tire',
    'slash_cut': 'damaged_tire', 'bulge': 'damaged_tire', 'worn': 'damaged_tire',
}
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
KAGGLE_DIR = Path('/content/kaggle_raw')

def collect_images(kaggle_dir):
    collected = defaultdict(list)
    for subdir in sorted(Path(kaggle_dir).iterdir()):
        if not subdir.is_dir(): continue
        cat = subdir.name.lower()
        cls = CATEGORY_MAP.get(cat)
        if cls is None:
            for k in CATEGORY_MAP:
                if k in cat: cls = CATEGORY_MAP[k]; break
        if cls is None:
            print(f'  ⚠ No mapeado: {cat}'); continue
        imgs = [p for p in subdir.rglob('*') if p.suffix.lower() in VALID_EXTS]
        collected[cls].extend(imgs)
        print(f'  {cat:20s} → {cls:15s}: {len(imgs):5d} imgs')
    return dict(collected)

def split_and_copy(collected, out_dir, val=0.15, test=0.15, seed=SEED):
    out = Path(out_dir)
    for sp in ['train','val','test']:
        for cls in collected: (out/sp/cls).mkdir(parents=True, exist_ok=True)
    stats = {}
    for cls, paths in collected.items():
        random.shuffle(paths)
        tr, tmp = train_test_split(paths, test_size=val+test, random_state=seed)
        vl, ts  = train_test_split(tmp, test_size=test/(val+test), random_state=seed)
        for sp, ps in [('train',tr),('val',vl),('test',ts)]:
            for src in ps:
                dst = out/sp/cls/src.name
                if dst.exists(): dst = out/sp/cls/f'{src.stem}_{id(src)}{src.suffix}'
                shutil.copy2(src, dst)
        stats[cls] = {'train':len(tr),'val':len(vl),'test':len(ts)}
        print(f'  {cls}: train={len(tr)} val={len(vl)} test={len(ts)}')
    return stats

print('Escaneando dataset...')
collected = collect_images(KAGGLE_DIR)
total = sum(len(v) for v in collected.values())
print(f'\nTotal imágenes: {total}')

print('\nDividiendo en train/val/test...')
stats = split_and_copy(collected, DATA_DIR)
print('\n✅ Dataset preparado.')

## 📊 Celda 4 — Análisis del Desbalance de Clases

In [ ]:
def analyze_and_plot_distribution(data_dir, splits=('train','val','test')):
    tf = transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.ToTensor()])
    report = {}
    fig, axes = plt.subplots(1, len(splits), figsize=(5*len(splits), 4))
    palette = ['#2196F3','#F44336','#4CAF50','#FF9800']

    for ax, split in zip(axes, splits):
        path = Path(data_dir)/split
        if not path.exists(): continue
        ds = datasets.ImageFolder(str(path), tf)
        counts = np.bincount(ds.targets)
        classes = ds.classes
        ratio = counts.max()/counts.min()
        report[split] = {'classes':classes,'counts':counts.tolist(),
                         'ratio':round(float(ratio),2)}
        bars = ax.bar(classes, counts, color=palette[:len(classes)], edgecolor='white', lw=1.5)
        for bar,cnt in zip(bars,counts):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+3,
                    f'{cnt}\n({100*cnt/counts.sum():.0f}%)',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
        ax.set_title(f'{split.capitalize()}\nRatio desbalance: {ratio:.1f}:1',
                     fontsize=11, fontweight='bold')
        ax.set_ylabel('Imágenes'); ax.set_ylim(0, counts.max()*1.2)
        ax.grid(axis='y', alpha=0.3)
        print(f'[{split}] {dict(zip(classes,counts))} | Ratio: {ratio:.2f}:1')
        if ratio > 2:
            print(f'  ⚠ Desbalance significativo → usar WeightedSampler + FocalLoss')

    plt.suptitle('Distribución de Clases por Split', fontsize=13, fontweight='bold')
    plt.tight_layout()
    (RESULTS/'imbalance').mkdir(exist_ok=True)
    plt.savefig(RESULTS/'imbalance/class_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    return report

dist_report = analyze_and_plot_distribution(DATA_DIR)
print('\n✅ Análisis de desbalance completado.')

## 🏗️ Celda 5 — Arquitecturas CNN

In [ ]:
# ════════════════════════════════════════════════════════════════════
#  BLOQUE 1: Módulos base
# ════════════════════════════════════════════════════════════════════
class ConvBnRelu(nn.Module):
    def __init__(self, in_ch, out_ch, ks=3, stride=1, pad=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, ks, stride=stride, padding=pad, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class ResBlock(nn.Module):
    """Bloque residual de 2 capas con shortcut de proyección opcional."""
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch))
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        return self.relu(self.bn2(self.conv2(out)) + self.shortcut(x))

class SEBlock(nn.Module):
    """Squeeze-and-Excitation: atención adaptativa de canal para texturas."""
    def __init__(self, ch, r=16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(ch, ch//r, bias=False), nn.ReLU(inplace=True),
            nn.Linear(ch//r, ch, bias=False), nn.Sigmoid())
    def forward(self, x): return x * self.se(x).unsqueeze(-1).unsqueeze(-1)

# ════════════════════════════════════════════════════════════════════
#  BLOQUE 2: CustomCNN — diseñada desde cero
# ════════════════════════════════════════════════════════════════════
class CustomCNN(nn.Module):
    """
    CNN para reconocimiento de textura de llantas.
    4 etapas: 64→128→256→512 canales con ResBlocks + SEBlocks.
    ~6.5M parámetros entrenables.
    """
    def __init__(self, num_classes=2):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1))          # 224→56
        self.stage1 = nn.Sequential(ResBlock(32,64),  SEBlock(64))
        self.stage2 = nn.Sequential(ResBlock(64,128,2), ResBlock(128,128), SEBlock(128))
        self.stage3 = nn.Sequential(ResBlock(128,256,2), ResBlock(256,256), SEBlock(256))
        self.stage4 = nn.Sequential(ResBlock(256,512,2), SEBlock(512))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(512,256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Dropout(0.5), nn.Linear(256, num_classes))
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        for s in [self.stage1, self.stage2, self.stage3, self.stage4]: x = s(x)
        return self.classifier(self.gap(x))

# ════════════════════════════════════════════════════════════════════
#  BLOQUE 3: Focal Loss
# ════════════════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    """FL(p_t) = -α_t·(1−p_t)^γ·log(p_t)  |  Lin et al., ICCV 2017"""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, reduction='none')
        p_t = torch.exp(-ce)
        at  = self.alpha * targets + (1-self.alpha)*(1-targets)
        return (at.float() * (1-p_t)**self.gamma * ce).mean()

# ════════════════════════════════════════════════════════════════════
#  BLOQUE 4: Construcción de modelos
# ════════════════════════════════════════════════════════════════════
def build_model(arch, num_classes=2, freeze_backbone=True):
    if arch == 'custom_cnn':
        return CustomCNN(num_classes).to(DEVICE)
    elif arch == 'resnet50':
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        if freeze_backbone:
            for n,p in m.named_parameters():
                if 'layer4' not in n and 'fc' not in n: p.requires_grad_(False)
        m.fc = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(m.fc.in_features, 256),
            nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, num_classes))
        return m.to(DEVICE)
    elif arch == 'efficientnet_b0':
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        if freeze_backbone:
            for n,p in m.named_parameters():
                if not any(x in n for x in ['features.7','features.8','classifier']):
                    p.requires_grad_(False)
        m.classifier = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(m.classifier[1].in_features, num_classes))
        return m.to(DEVICE)
    raise ValueError(f'Arquitectura desconocida: {arch}')

# Verificación
for arch in ['custom_cnn']:
    m = build_model(arch)
    x = torch.randn(2,3,224,224).to(DEVICE)
    out = m(x)
    tr = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{arch:20s} → out={tuple(out.shape)} | trainable params: {tr:,}')
del m, x, out
print('\n✅ Arquitecturas definidas.')

## 🔄 Celda 6 — Dataloaders y Transformaciones

In [ ]:
def get_transforms(augment=True):
    norm = transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    if augment:
        train_tf = transforms.Compose([
            transforms.Resize((IMG_SIZE+32, IMG_SIZE+32)),
            transforms.RandomCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.3),
            transforms.RandomRotation(30),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
            transforms.ToTensor(), norm,
            transforms.RandomErasing(p=0.2, scale=(0.02,0.15))])
    else:
        train_tf = transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),
                                       transforms.ToTensor(), norm])
    val_tf = transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),
                                  transforms.ToTensor(), norm])
    return train_tf, val_tf

def get_dataloaders(augment=True, use_weighted_sampler=True):
    train_tf, val_tf = get_transforms(augment)
    train_ds = datasets.ImageFolder(str(DATA_DIR/'train'), train_tf)
    val_ds   = datasets.ImageFolder(str(DATA_DIR/'val'),   val_tf)
    test_ds  = datasets.ImageFolder(str(DATA_DIR/'test'),  val_tf)

    counts = np.bincount(train_ds.targets).astype(float)
    cw = 1.0/counts
    sw = torch.tensor([cw[t] for t in train_ds.targets])

    if use_weighted_sampler:
        sampler = WeightedRandomSampler(sw, len(sw), replacement=True)
        train_loader = DataLoader(train_ds, BATCH_SIZE, sampler=sampler,
                                  num_workers=2, pin_memory=True)
    else:
        train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,
                                  num_workers=2, pin_memory=True)

    val_loader  = DataLoader(val_ds,  BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    classes = train_ds.classes
    print(f'Clases       : {classes}')
    print(f'Train        : {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
    print(f'Distribución : {dict(zip(classes, counts.astype(int)))}')
    return train_loader, val_loader, test_loader, classes, cw

train_loader, val_loader, test_loader, CLASS_NAMES, CLASS_WEIGHTS = get_dataloaders()
NUM_CLASSES = len(CLASS_NAMES)
print('\n✅ Dataloaders listos.')

## 🧰 Celda 7 — Utilidades de Entrenamiento

In [ ]:
class EarlyStopping:
    def __init__(self, patience=7, delta=1e-4, path='best.pth'):
        self.patience=patience; self.delta=delta; self.path=path
        self.counter=0; self.best=np.inf; self.early_stop=False
    def __call__(self, val_loss, model):
        if val_loss < self.best - self.delta:
            self.best=val_loss; self.counter=0
            torch.save(model.state_dict(), self.path)
        else:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop=True

def train_epoch(model, loader, optimizer, criterion, scaler=None):
    model.train()
    total_loss=0; correct=0; n=0
    for imgs,labels in loader:
        imgs,labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        if scaler:
            with torch.autocast(device_type='cuda'):
                logits=model(imgs); loss=criterion(logits,labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            logits=model(imgs); loss=criterion(logits,labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total_loss+=loss.item()*imgs.size(0)
        correct+=(logits.argmax(1)==labels).sum().item(); n+=imgs.size(0)
    return total_loss/n, correct/n

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss=0; preds=[]; labels=[]; probs=[]
    for imgs,lbs in loader:
        imgs,lbs = imgs.to(DEVICE), lbs.to(DEVICE)
        logits=model(imgs); loss=criterion(logits,lbs)
        total_loss+=loss.item()*imgs.size(0)
        p=torch.softmax(logits,1)[:,1]
        preds.extend(logits.argmax(1).cpu().numpy())
        labels.extend(lbs.cpu().numpy())
        probs.extend(p.cpu().numpy())
    n=len(labels)
    return {
        'loss':      total_loss/n,
        'accuracy':  accuracy_score(labels,preds),
        'precision': precision_score(labels,preds, zero_division=0),
        'recall':    recall_score(labels,preds, zero_division=0),
        'f1':        f1_score(labels,preds, zero_division=0),
        'auc_roc':   roc_auc_score(labels,probs),
    }, preds, labels, probs

def run_training(arch, loss_fn='focal', augment=True, use_sampler=True,
                 lr=1e-4, epochs=EPOCHS, tag=None):
    set_seed()
    tag = tag or arch
    out = RESULTS/tag; out.mkdir(parents=True, exist_ok=True)

    loader_tr, loader_vl, loader_ts, classes, cw = get_dataloaders(augment, use_sampler)
    model = build_model(arch)
    tr_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n{"="*55}')
    print(f' {arch} | loss={loss_fn} | aug={augment} | sampler={use_sampler} | lr={lr}')
    print(f' Parámetros entrenables: {tr_params:,}')
    print(f'{"="*55}')

    if loss_fn == 'focal':
        criterion = FocalLoss(0.25, 2.0)
    else:
        w = torch.tensor(cw/cw.sum()*NUM_CLASSES, dtype=torch.float).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=w)

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.cuda.amp.GradScaler() if DEVICE.type=='cuda' else None
    es = EarlyStopping(PATIENCE, path=str(out/'best.pth'))

    hist = {'tr_loss':[],'vl_loss':[],'tr_acc':[],'vl_acc':[],'vl_f1':[]}

    for ep in range(1, epochs+1):
        tl,ta = train_epoch(model, loader_tr, optimizer, criterion, scaler)
        vm,_,_,_ = evaluate(model, loader_vl, criterion)
        scheduler.step()
        for k,v in [('tr_loss',tl),('vl_loss',vm['loss']),
                    ('tr_acc',ta),('vl_acc',vm['accuracy']),('vl_f1',vm['f1'])]:
            hist[k].append(v)
        print(f'Ep {ep:02d}/{epochs} | tr_loss={tl:.4f} tr_acc={ta:.4f} '
              f'| vl_f1={vm["f1"]:.4f} vl_auc={vm["auc_roc"]:.4f}', end=' ')
        es(vm['loss'], model)
        print('✓' if not es.early_stop else '⏹')
        if es.early_stop: print(f'Early stopping en ep {ep}'); break

    model.load_state_dict(torch.load(out/'best.pth', map_location=DEVICE))
    test_m, test_p, test_l, test_pr = evaluate(model, loader_ts, criterion)
    print(f'\n[TEST] {tag}')
    print(classification_report(test_l, test_p, target_names=classes, digits=4))
    print(f'  AUC-ROC: {test_m["auc_roc"]:.4f}')

    # Guardar curvas
    ep_range = range(1, len(hist['tr_loss'])+1)
    fig,axes = plt.subplots(1,2,figsize=(12,4))
    axes[0].plot(ep_range,hist['tr_loss'],label='Train'); axes[0].plot(ep_range,hist['vl_loss'],label='Val')
    axes[0].set_title(f'Pérdida — {tag}'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(ep_range,hist['tr_acc'],label='Train acc'); axes[1].plot(ep_range,hist['vl_acc'],label='Val acc')
    axes[1].plot(ep_range,hist['vl_f1'],'--',label='Val F1'); axes[1].set_title(f'Accuracy & F1 — {tag}')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(out/'training_curves.png',dpi=130,bbox_inches='tight'); plt.show()

    # Guardar matriz de confusión
    cm_arr = confusion_matrix(test_l, test_p)
    fig,ax = plt.subplots(figsize=(5,4))
    sns.heatmap(cm_arr, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_xlabel('Predicción'); ax.set_ylabel('Real')
    ax.set_title(f'Matriz de Confusión — {tag}')
    plt.tight_layout(); plt.savefig(out/'confusion_matrix.png',dpi=130,bbox_inches='tight'); plt.show()

    with open(out/'test_metrics.json','w') as f:
        json.dump({k:round(float(v),4) for k,v in test_m.items()}, f, indent=2)

    return model, test_m, test_l, test_pr, hist

print('✅ Funciones de entrenamiento definidas.')

## 🚀 Celda 8 — Entrenamiento: CNN desde Cero

In [ ]:
model_cnn, metrics_cnn, labels_cnn, probs_cnn, hist_cnn = run_training(
    arch='custom_cnn',
    loss_fn='ce',
    augment=True,
    use_sampler=True,
    lr=1e-4,
    epochs=EPOCHS,
    tag='custom_cnn'
)

## 🚀 Celda 9 — Entrenamiento: ResNet-50 con Fine-tuning

In [ ]:
model_r50, metrics_r50, labels_r50, probs_r50, hist_r50 = run_training(
    arch='resnet50',
    loss_fn='focal',
    augment=True,
    use_sampler=True,
    lr=1e-4,
    epochs=EPOCHS,
    tag='resnet50'
)

## 🚀 Celda 10 — Entrenamiento: EfficientNet-B0 con Fine-tuning

In [ ]:
model_eff, metrics_eff, labels_eff, probs_eff, hist_eff = run_training(
    arch='efficientnet_b0',
    loss_fn='focal',
    augment=True,
    use_sampler=True,
    lr=1e-4,
    epochs=EPOCHS,
    tag='efficientnet_b0'
)

## 📈 Celda 11 — Evaluación Comparativa

In [ ]:
ALL_RESULTS = {
    'CustomCNN (scratch)':     {'metrics': metrics_cnn, 'labels': labels_cnn, 'probs': probs_cnn},
    'ResNet-50 (FT)':          {'metrics': metrics_r50, 'labels': labels_r50, 'probs': probs_r50},
    'EfficientNet-B0 (FT)':    {'metrics': metrics_eff, 'labels': labels_eff, 'probs': probs_eff},
}

# ── Curvas ROC ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_map = {'CustomCNN (scratch)':'#F44336','ResNet-50 (FT)':'#2196F3',
              'EfficientNet-B0 (FT)':'#4CAF50'}

for name, data in ALL_RESULTS.items():
    fpr, tpr, _ = roc_curve(data['labels'], data['probs'])
    auc_sc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc_sc:.4f})',
                 color=colors_map[name], lw=2)
    prec, rec, _ = precision_recall_curve(data['labels'], data['probs'])
    ap = average_precision_score(data['labels'], data['probs'])
    axes[1].plot(rec, prec, label=f'{name} (AP={ap:.4f})',
                 color=colors_map[name], lw=2)

axes[0].plot([0,1],[0,1],'k--',alpha=0.4); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('Curvas ROC', fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Curvas Precision-Recall', fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
(RESULTS/'comparison').mkdir(exist_ok=True)
plt.savefig(RESULTS/'comparison/roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Tabla resumen ─────────────────────────────────────────────────────────────
import pandas as pd
rows = []
for name, data in ALL_RESULTS.items():
    m = data['metrics']
    rows.append({'Modelo': name,
                 'Accuracy':  f"{m['accuracy']:.4f}",
                 'Precision': f"{m['precision']:.4f}",
                 'Recall':    f"{m['recall']:.4f}",
                 'F1-Score':  f"{m['f1']:.4f}",
                 'AUC-ROC':   f"{m['auc_roc']:.4f}"})
df = pd.DataFrame(rows).set_index('Modelo')
print('\n' + '='*60)
print('  COMPARACIÓN FINAL — CONJUNTO DE PRUEBA')
print('='*60)
print(df.to_string())
df.to_csv(RESULTS/'comparison/metrics_table.csv')
print('\n✅ Evaluación comparativa completada.')

## 🔬 Celda 12 — Estudio de Ablación

In [ ]:
# Ablación sobre ResNet-50 (4 factores × 2-3 variantes)
ABLATION_GRID = [
    # (tag,            arch,       loss,    aug,   sampler, lr   )
    ('A_aug_ON',      'resnet50', 'focal', True,  True,  1e-4),
    ('A_aug_OFF',     'resnet50', 'focal', False, True,  1e-4),
    ('B_focal',       'resnet50', 'focal', True,  True,  1e-4),
    ('B_ce_weights',  'resnet50', 'ce',    True,  True,  1e-4),
    ('C_sampler_ON',  'resnet50', 'focal', True,  True,  1e-4),
    ('C_sampler_OFF', 'resnet50', 'focal', True,  False, 1e-4),
    ('D_lr_1e3',      'resnet50', 'focal', True,  True,  1e-3),
    ('D_lr_1e4',      'resnet50', 'focal', True,  True,  1e-4),
    ('D_lr_5e5',      'resnet50', 'focal', True,  True,  5e-5),
]

ablation_results = []
for tag, arch, loss, aug, samp, lr in ABLATION_GRID:
    print(f'\n>>> Ablación: {tag}')
    _, m, _, _, _ = run_training(arch, loss, aug, samp, lr,
                                  epochs=15, tag=f'ablation/{tag}')
    ablation_results.append({'Variante':tag, 'F1':round(m['f1'],4),
                              'Recall':round(m['recall'],4), 'AUC':round(m['auc_roc'],4)})

df_abl = pd.DataFrame(ablation_results).set_index('Variante')
print('\n' + '='*50)
print('  RESULTADOS DE ABLACIÓN')
print('='*50)
print(df_abl.to_string())

# Visualización
groups = [('Augmentación',['A_aug_ON','A_aug_OFF']),
          ('Loss Function',['B_focal','B_ce_weights']),
          ('Class Sampler',['C_sampler_ON','C_sampler_OFF']),
          ('Learning Rate',['D_lr_1e3','D_lr_1e4','D_lr_5e5'])]

fig, axes = plt.subplots(1,4, figsize=(18,5))
for ax, (grp_name, variants) in zip(axes, groups):
    sub = df_abl.loc[variants]
    x = np.arange(len(variants)); w=0.25
    for i,(col,c) in enumerate(zip(['F1','Recall','AUC'],['#2196F3','#4CAF50','#FF5722'])):
        bars = ax.bar(x+i*w, sub[col], w, label=col, color=c, alpha=0.85)
        for b in bars:
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
                    f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    ax.set_xticks(x+w); ax.set_xticklabels(variants, rotation=15, ha='right', fontsize=8)
    ax.set_ylim(0,1.1); ax.set_title(grp_name, fontweight='bold'); ax.legend(fontsize=7); ax.grid(axis='y',alpha=0.3)
plt.suptitle('Estudio de Ablación — ResNet-50', fontsize=13, fontweight='bold')
plt.tight_layout()
(RESULTS/'ablation').mkdir(exist_ok=True)
plt.savefig(RESULTS/'ablation/ablation_plot.png', dpi=150, bbox_inches='tight')
plt.show()
df_abl.to_csv(RESULTS/'ablation/ablation_results.csv')
print('\n✅ Estudio de ablación completado.')

## 🔍 Celda 13 — Grad-CAM: Interpretabilidad

In [ ]:
# ── Implementación Grad-CAM ───────────────────────────────────────────────────
class GradCAM:
    """Selvaraju et al., ICCV 2017."""
    def __init__(self, model, target_layer):
        self.model = model; self._grad = None; self._act = None
        target_layer.register_forward_hook(lambda m,i,o: setattr(self,'_act',o.detach()))
        target_layer.register_full_backward_hook(lambda m,gi,go: setattr(self,'_grad',go[0].detach()))

    @torch.enable_grad()
    def __call__(self, img_t, cls=None):
        self.model.eval()
        img_t = img_t.requires_grad_(True)
        logits = self.model(img_t)
        cls = cls if cls is not None else logits.argmax(1).item()
        self.model.zero_grad()
        logits[0,cls].backward()
        w   = self._grad.mean([2,3], keepdim=True)
        cam = F.relu((w * self._act).sum(1, keepdim=True)).squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, cls

def overlay_cam(img_pil, cam, alpha=0.45):
    h,w = np.array(img_pil).shape[:2]
    cam_r = np.array(Image.fromarray((cam*255).astype(np.uint8)).resize((w,h),Image.BILINEAR))/255
    heat  = (cm.jet(cam_r)[:,:,:3]*255).astype(np.uint8)
    orig  = np.array(img_pil.convert('RGB')).astype(float)
    return Image.fromarray(np.clip((1-alpha)*orig + alpha*heat, 0,255).astype(np.uint8))

MEAN = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
STD  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)

def denorm(t): return (t.cpu()*STD+MEAN).clamp(0,1)

def show_gradcam(model, target_layer, loader, n_per_class=3):
    gcam = GradCAM(model, target_layer)
    collected = {i:[] for i in range(NUM_CLASSES)}
    for imgs,labels in loader:
        for img,lbl in zip(imgs,labels):
            l = lbl.item()
            if len(collected[l])>=n_per_class: continue
            img_t = img.unsqueeze(0).to(DEVICE)
            cam,pred = gcam(img_t)
            pil = transforms.ToPILImage()(denorm(img))
            collected[l].append({'orig':pil,'overlay':overlay_cam(pil,cam),'cam':cam,'pred':pred,'true':l})
        if all(len(v)>=n_per_class for v in collected.values()): break

    rows = NUM_CLASSES*n_per_class
    fig,axes = plt.subplots(rows,3,figsize=(12,4*rows))
    if rows==1: axes=axes[np.newaxis,:]
    r=0
    for cls_i,samples in collected.items():
        for s in samples:
            axes[r,0].imshow(s['orig'])
            axes[r,0].set_title(f"GT:{CLASS_NAMES[s['true']]} | Pred:{CLASS_NAMES[s['pred']]}",fontsize=8)
            axes[r,0].axis('off')
            axes[r,1].imshow(s['overlay'])
            axes[r,1].set_title('Grad-CAM Overlay',fontsize=8); axes[r,1].axis('off')
            axes[r,2].imshow(s['cam'],cmap='jet')
            axes[r,2].set_title('Activation Map',fontsize=8); axes[r,2].axis('off')
            r+=1
    plt.suptitle('Grad-CAM — Interpretabilidad', fontsize=13, fontweight='bold')
    plt.tight_layout()
    (RESULTS/'gradcam').mkdir(exist_ok=True)
    plt.savefig(RESULTS/'gradcam/gradcam_analysis.png',dpi=150,bbox_inches='tight'); plt.show()

# Usar EfficientNet-B0 (mejor modelo) con su última capa convolucional
target_eff = model_eff.features[8][0]  # última conv de EfficientNet-B0
show_gradcam(model_eff, target_eff, test_loader, n_per_class=3)

# También para CustomCNN
target_cnn = model_cnn.stage4[0].conv2
show_gradcam(model_cnn, target_cnn, test_loader, n_per_class=3)
print('\n✅ Grad-CAM completado.')

## ❌ Celda 14 — Análisis Cualitativo de Fallos

In [ ]:
def analyze_failures(model, target_layer, loader, n=5):
    gcam = GradCAM(model, target_layer)
    failures = []
    model.eval()
    with torch.no_grad():
        for imgs,labels in loader:
            imgs_d = imgs.to(DEVICE)
            logits = model(imgs_d)
            probs  = torch.softmax(logits,1)
            preds  = logits.argmax(1)
            for i in range(len(labels)):
                if preds[i].item()!=labels[i].item():
                    failures.append({'img':imgs[i],'true':labels[i].item(),
                                     'pred':preds[i].item(),'conf':probs[i,preds[i]].item()})
            if len(failures)>=n*3: break

    failures = sorted(failures, key=lambda x: x['conf'], reverse=True)[:n]
    fig,axes = plt.subplots(len(failures),3,figsize=(12,4*len(failures)))
    if len(failures)==1: axes=axes[np.newaxis,:]

    print(f'\n{"="*55}')
    print(f'  ANÁLISIS DE FALLOS ({n} más confiados mal clasificados)')
    print(f'{"═"*55}')

    for ri,fail in enumerate(failures):
        img_t = fail['img'].unsqueeze(0).to(DEVICE)
        cam,_ = gcam(img_t)
        pil = transforms.ToPILImage()(denorm(fail['img']))
        ov  = overlay_cam(pil, cam)
        axes[ri,0].imshow(pil)
        axes[ri,0].set_title(
            f"GT: {CLASS_NAMES[fail['true']]}  →  Pred: {CLASS_NAMES[fail['pred']]}  "
            f"(conf={fail['conf']:.2f})", fontsize=9, color='red')
        axes[ri,0].axis('off')
        axes[ri,1].imshow(ov); axes[ri,1].set_title('Grad-CAM Overlay',fontsize=9); axes[ri,1].axis('off')
        axes[ri,2].imshow(cam,cmap='jet'); axes[ri,2].set_title('Activation Map',fontsize=9); axes[ri,2].axis('off')
        err = 'FN (daño no detectado)' if fail['true']==1 else 'FP (buena clasificada dañada)'
        print(f'  #{ri+1} | {err} | conf={fail["conf"]:.3f}')

    plt.suptitle(f'Análisis de Fallos — {n} ejemplos más confiados',fontsize=13,fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS/'gradcam/failure_analysis.png',dpi=150,bbox_inches='tight'); plt.show()

analyze_failures(model_eff, target_eff, test_loader, n=5)
print('\n✅ Análisis de fallos completado.')

## 📋 Celda 15 — Análisis de Estrategias de Mitigación de Desbalance

In [ ]:
# Comparar 4 estrategias de mitigación usando ResNet-50
MITIGATION_GRID = [
    ('Sin mitigación',          'resnet50', 'ce',    False, False),
    ('CE + pesos de clase',     'resnet50', 'ce',    True,  False),
    ('WeightedRandomSampler',   'resnet50', 'ce',    False, True ),
    ('Focal Loss',              'resnet50', 'focal', True,  True ),
]

mit_results = []
for name, arch, loss, aug, samp in MITIGATION_GRID:
    tag = name.replace(' ','_').replace('+','').lower()
    print(f'\n>>> Estrategia: {name}')
    _,m,_,_,_ = run_training(arch, loss, aug, samp, 1e-4,
                               epochs=15, tag=f'mitigation/{tag}')
    mit_results.append({'Estrategia':name, 'Accuracy':round(m['accuracy'],4),
                        'Precision':round(m['precision'],4), 'Recall (damaged)':round(m['recall'],4),
                        'F1':round(m['f1'],4), 'AUC-ROC':round(m['auc_roc'],4)})

df_mit = pd.DataFrame(mit_results).set_index('Estrategia')
print('\n' + '='*60)
print('  COMPARACIÓN DE ESTRATEGIAS DE MITIGACIÓN')
print('='*60)
print(df_mit.to_string())

# Gráfico comparativo
fig, ax = plt.subplots(figsize=(12,5))
names = df_mit.index.tolist()
metrics_cols = ['Recall (damaged)','F1','AUC-ROC']
cols_c = ['#F44336','#2196F3','#4CAF50']
x = np.arange(len(names)); w=0.25
for i,(mc,c) in enumerate(zip(metrics_cols,cols_c)):
    bars = ax.bar(x+i*w, df_mit[mc], w, label=mc, color=c, alpha=0.85)
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.004,
                f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x+w); ax.set_xticklabels(names, rotation=10, ha='right', fontsize=9)
ax.set_ylim(0,1.1); ax.set_title('Comparación de Estrategias de Mitigación del Desbalance',
                                   fontsize=12, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
(RESULTS/'imbalance').mkdir(exist_ok=True)
plt.savefig(RESULTS/'imbalance/mitigation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
df_mit.to_csv(RESULTS/'imbalance/mitigation_comparison.csv')
print('\n✅ Análisis de mitigación completado.')

## 💾 Celda 16 — Exportar Resultados a Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = Path('/content/drive/MyDrive/RNAPD_P1_results')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
shutil.copytree(str(RESULTS), str(DRIVE_OUT/'results'), dirs_exist_ok=True)

# También comprimir para descarga directa
shutil.make_archive('/content/RNAPD_P1_results', 'zip', str(RESULTS))
print('✅ Resultados exportados a Google Drive y disponibles para descarga.')
print(f'   Ruta Drive: {DRIVE_OUT}')

## ✅ Resumen Final

| Entregable | Archivo generado |
|---|---|
| Modelos entrenados | `results/{arch}/best.pth` |
| Curvas de entrenamiento | `results/{arch}/training_curves.png` |
| Matrices de confusión | `results/{arch}/confusion_matrix.png` |
| Métricas test | `results/{arch}/test_metrics.json` |
| Curvas ROC + PR | `results/comparison/roc_pr_curves.png` |
| Tabla comparativa | `results/comparison/metrics_table.csv` |
| Ablación | `results/ablation/ablation_plot.png` + `.csv` |
| Distribución clases | `results/imbalance/class_distribution.png` |
| Mitigación desbalance | `results/imbalance/mitigation_comparison.png` |
| Grad-CAM | `results/gradcam/gradcam_analysis.png` |
| Análisis de fallos | `results/gradcam/failure_analysis.png` |